# Lecture 12: Backpropagation from Scratch on a Single Neuron

**Source:** [Lecture 12 - Backpropagation from scratch on a single neuron](https://www.youtube.com/watch?v=iE1lccrHfok) (Vizuara)

We implement forward pass (sum → ReLU → squared loss), backward pass (chain rule), and gradient descent updates for **one neuron** with 3 inputs. See **Lecture12_backprop_single_neuron_lesson.md** for the full lesson and diagrams.

---
## 1. Setup: inputs, weights, bias, target, learning rate

Same numerical example as in the video.

In [3]:
import numpy as np

# Inputs (fixed from data)
x = np.array([1.0, -2.0, 3.0])  # x0, x1, x2

# Weights and bias (initial; will be updated)
w = np.array([-3.0, -1.0, 2.0])  # w0, w1, w2
b = 1.0

# Target output (we want the neuron output to approach 0)
target = 0.0

# Learning rate (step size for gradient descent)
learning_rate = 0.01

print("Inputs x:", x)
print("Weights w:", w)
print("Bias b:", b)
print("Target:", target)
print("Learning rate:", learning_rate)

Inputs x: [ 1. -2.  3.]
Weights w: [-3. -1.  2.]
Bias b: 1.0
Target: 0.0
Learning rate: 0.01


---
## 2. Forward pass (step-by-step)

Compute: **sum** = x·w + b → **Z** = ReLU(sum) → **loss** = (Z − target)² = Z²

In [4]:
def relu(a):
    return np.maximum(0, a)

def relu_derivative(a):
    return (a > 0).astype(float)

# Forward pass (one step at a time)
weighted_sum = np.dot(x, w) + b
z = relu(weighted_sum)
loss = (z - target) ** 2

print("Step 1 — Weighted sum (x·w + b):", weighted_sum)
print("Step 2 — ReLU(sum):", z)
print("Step 3 — Loss (Z - target)²:", loss)

Step 1 — Weighted sum (x·w + b): 6.0
Step 2 — ReLU(sum): 6.0
Step 3 — Loss (Z - target)²: 36.0


---
## 3. Backward pass (gradients by chain rule)

Compute:
- d_loss_d_z = 2 * (Z - target)
- d_z_d_sum = ReLU'(sum)
- d_loss_d_weights = d_loss_d_z * d_z_d_sum * x
- d_loss_d_bias = d_loss_d_z * d_z_d_sum

In [5]:
# Gradient of loss w.r.t. output Z: d(Z²)/dZ = 2Z (with target=0)
d_loss_d_z = 2 * (z - target)

# Gradient of ReLU w.r.t. its input (sum): 1 if sum>0 else 0
d_z_d_sum = relu_derivative(weighted_sum)

# Chain rule: dL/dw = (dL/dZ) * (dZ/d_sum) * (d_sum/d(x_i*w_i)) * (d(x_i*w_i)/dw_i)
# d_sum/d(x_i*w_i) = 1, d(x_i*w_i)/dw_i = x_i  =>  dL/dw_i = d_loss_d_z * d_z_d_sum * x_i
d_loss_d_weights = d_loss_d_z * d_z_d_sum * x

# dL/db = d_loss_d_z * d_z_d_sum * 1
d_loss_d_bias = d_loss_d_z * d_z_d_sum

print("d_loss_d_z (2*(Z-target)):", d_loss_d_z)
print("d_z_d_sum (ReLU'):", d_z_d_sum)
print("d_loss_d_weights:", d_loss_d_weights)
print("d_loss_d_bias:", d_loss_d_bias)

d_loss_d_z (2*(Z-target)): 12.0
d_z_d_sum (ReLU'): 1.0
d_loss_d_weights: [ 12. -24.  36.]
d_loss_d_bias: 12.0


---
## 4. One gradient descent step (iteration 1)

Update: **w := w − η * d_loss_d_weights**, **b := b − η * d_loss_d_bias**. Then recompute loss.

In [6]:
print("Before update:")
print("  w =", w)
print("  b =", b)
print("  loss =", loss)

w = w - learning_rate * d_loss_d_weights
b = b - learning_rate * d_loss_d_bias

weighted_sum_new = np.dot(x, w) + b
z_new = relu(weighted_sum_new)
loss_new = (z_new - target) ** 2

print("After one update:")
print("  w =", w)
print("  b =", b)
print("  new sum =", weighted_sum_new)
print("  new Z =", z_new)
print("  new loss =", loss_new)

Before update:
  w = [-3. -1.  2.]
  b = 1.0
  loss = 36.0
After one update:
  w = [-3.12 -0.76  1.64]
  b = 0.88
  new sum = 4.2
  new Z = 4.2
  new loss = 17.64


---
## 5. Full training loop (200 iterations)

Repeatedly: forward pass → backward pass → update. Print every 50 iterations and at the end.

In [7]:
# Reset to initial values for the full loop
w = np.array([-3.0, -1.0, 2.0])
b = 1.0

n_iters = 200
print_every = 50

for i in range(n_iters):
    # Forward
    weighted_sum = np.dot(x, w) + b
    z = relu(weighted_sum)
    loss = (z - target) ** 2

    # Backward
    d_loss_d_z = 2 * (z - target)
    d_z_d_sum = relu_derivative(weighted_sum)
    d_loss_d_weights = d_loss_d_z * d_z_d_sum * x
    d_loss_d_bias = d_loss_d_z * d_z_d_sum

    # Update
    w = w - learning_rate * d_loss_d_weights
    b = b - learning_rate * d_loss_d_bias

    if (i + 1) % print_every == 0 or i == 0:
        print(f"Iter {i+1:3d}  loss = {loss:.6f}  Z = {z:.6f}  w = {w}  b = {b:.6f}")

print("\nFinal:")
print("  w =", w)
print("  b =", b)
print("  final loss =", loss)

Iter   1  loss = 36.000000  Z = 6.000000  w = [-3.12 -0.76  1.64]  b = 0.880000
Iter  50  loss = 0.000000  Z = 0.000000  w = [-3.39999999 -0.20000001  0.80000002]  b = 0.600000
Iter 100  loss = 0.000000  Z = 0.000000  w = [-3.4 -0.2  0.8]  b = 0.600000
Iter 150  loss = 0.000000  Z = 0.000000  w = [-3.4 -0.2  0.8]  b = 0.600000
Iter 200  loss = 0.000000  Z = 0.000000  w = [-3.4 -0.2  0.8]  b = 0.600000

Final:
  w = [-3.4 -0.2  0.8]
  b = 0.6000000000000005
  final loss = 0.0


---
## 6. Iteration-by-iteration breakdown (first 5 steps)

Same loop but with **every** iteration printed so you can see the numbers change step by step.

In [8]:
w = np.array([-3.0, -1.0, 2.0])
b = 1.0

for i in range(5):
    weighted_sum = np.dot(x, w) + b
    z = relu(weighted_sum)
    loss = (z - target) ** 2

    d_loss_d_z = 2 * (z - target)
    d_z_d_sum = relu_derivative(weighted_sum)
    d_loss_d_weights = d_loss_d_z * d_z_d_sum * x
    d_loss_d_bias = d_loss_d_z * d_z_d_sum

    print(f"--- Iteration {i+1} ---")
    print(f"  sum = {weighted_sum:.4f}, Z = {z:.4f}, loss = {loss:.4f}")
    print(f"  dL/dw = {d_loss_d_weights}, dL/db = {d_loss_d_bias}")

    w = w - learning_rate * d_loss_d_weights
    b = b - learning_rate * d_loss_d_bias
    print(f"  updated w = {w}, b = {b:.4f}")
    print()

--- Iteration 1 ---
  sum = 6.0000, Z = 6.0000, loss = 36.0000
  dL/dw = [ 12. -24.  36.], dL/db = 12.0
  updated w = [-3.12 -0.76  1.64], b = 0.8800

--- Iteration 2 ---
  sum = 4.2000, Z = 4.2000, loss = 17.6400
  dL/dw = [  8.4 -16.8  25.2], dL/db = 8.4
  updated w = [-3.204 -0.592  1.388], b = 0.7960

--- Iteration 3 ---
  sum = 2.9400, Z = 2.9400, loss = 8.6436
  dL/dw = [  5.88 -11.76  17.64], dL/db = 5.880000000000001
  updated w = [-3.2628 -0.4744  1.2116], b = 0.7372

--- Iteration 4 ---
  sum = 2.0580, Z = 2.0580, loss = 4.2354
  dL/dw = [ 4.116 -8.232 12.348], dL/db = 4.1160000000000005
  updated w = [-3.30396 -0.39208  1.08812], b = 0.6960

--- Iteration 5 ---
  sum = 1.4406, Z = 1.4406, loss = 2.0753
  dL/dw = [ 2.8812 -5.7624  8.6436], dL/db = 2.8812000000000006
  updated w = [-3.332772 -0.334456  1.001684], b = 0.6672



---
## 7. Summary

- **Forward:** sum = x·w + b → Z = ReLU(sum) → loss = (Z − target)².
- **Backward:** dL/dZ = 2(Z−target), ReLU' = 1 or 0, then multiply by inputs to get dL/dw; dL/db = dL/dZ × ReLU'.
- **Update:** w -= η * dL/dw, b -= η * dL/db.

This is the core of backprop for one neuron; layering many such neurons follows the same idea with more bookkeeping.